In [1]:
import pandas as pd
import numpy as np
import os
import glob
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
 

pd.set_option("display.max_columns", 100)

DATA_HOT_SCORE = Path("data/hotscore")
OUTPUT_DIR = Path("output/volatility")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def load_all_hotscore_files(directory=DATA_HOT_SCORE):
    files = sorted(glob.glob(str(directory / "hotscore_*.csv")))
    
    dfs = []
    
    for f in files:
        df = pd.read_csv(f)
        
        # extract date from filename
        date_str = os.path.basename(f).split("_")[1].replace(".csv", "")
        df["date"] = pd.to_datetime(date_str, format="%Y%m%d")
        
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

df = load_all_hotscore_files()

df = df.sort_values(["symbol", "date"])
df.head(5)

,symbol,HotScore,TrendScore,regularMarketPrice,regularMarketChangePercent,VolumeSpike,averageDailyVolume3Month,MomentumScore,VolumeScore,VolatilityScore,marketCap,date
0,AA,0.794401,0.520833,41.845,6.74745,0.940394,6727448.0,0.903646,0.802083,0.726562,1.083635e+10,2026-01-17
50,AA,0.794401,0.520833,41.845,6.74745,0.940394,6727448.0,0.903646,0.802083,0.726562,1.083635e+10,2026-01-17
100,AA,0.773989,0.557951,41.560,6.02041,1.039631,6727448.0,0.876011,0.749326,0.746631,1.076255e+10,2026-01-17
150,AA,0.789218,0.571429,41.570,6.04592,1.136707,6727448.0,0.881402,0.778976,0.754717,1.076513e+10,2026-01-17
200,AA,0.789218,0.571429,41.570,6.04592,1.136707,6727448.0,0.881402,0.778976,0.754717,1.076513e+10,2026-01-17


In [5]:
df.info()

<class 'pandas.DataFrame'>
Index: 712505 entries, 0 to 708639
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   symbol                      712505 non-null  str           
 1   HotScore                    710084 non-null  float64       
 2   TrendScore                  710084 non-null  float64       
 3   regularMarketPrice          710084 non-null  float64       
 4   regularMarketChangePercent  710084 non-null  float64       
 5   VolumeSpike                 710084 non-null  float64       
 6   averageDailyVolume3Month    710084 non-null  float64       
 7   MomentumScore               710084 non-null  float64       
 8   VolumeScore                 710084 non-null  float64       
 9   VolatilityScore             710084 non-null  float64       
 10  marketCap                   710083 non-null  float64       
 11  date                        712505 non-null  datetime64

In [6]:
price_stats = df.groupby("symbol").agg(
    min_price=("regularMarketPrice", "min"),
    max_price=("regularMarketPrice", "max"),
    mean_price=("regularMarketPrice", "mean")
)

price_stats["range_pct"] = (
    (price_stats["max_price"] - price_stats["min_price"]) 
    / price_stats["mean_price"]
)

In [7]:
df["direction"] = np.sign(df["regularMarketChangePercent"])

direction_changes = (
    df.groupby("symbol")["direction"]
    .apply(lambda x: (x.diff() != 0).sum())
    .to_frame(name="direction_changes")
)

In [8]:
metrics = df.groupby("symbol").agg(
    avg_volatility=("VolatilityScore", "mean"),
    max_volatility=("VolatilityScore", "max"),
    avg_momentum=("MomentumScore", "mean"),
    avg_volume=("VolumeScore", "mean"),
    avg_volume_spike=("VolumeSpike", "mean"),
    avg_change_pct=("regularMarketChangePercent", "mean")
)

In [9]:
final = price_stats.join(direction_changes).join(metrics)

# Normalize direction_changes
final["dir_change_norm"] = final["direction_changes"].rank(pct=True)

# Swing Score formula
final["swing_score"] = (
    final["range_pct"] * 0.35 +
    final["dir_change_norm"] * 0.30 +
    final["avg_volatility"] * 0.20 +
    final["avg_volume"] * 0.15
)

final = final.sort_values("swing_score", ascending=False)

In [13]:
swing_candidates = final[
    (final["avg_volatility"] > 0.6) &
    (final["avg_volume"] > 0.5) &
    (final["range_pct"] > 0.25)
].sort_values("swing_score", ascending=False)

top_swings = swing_candidates.head(20)

top_swings.head(10)

,min_price,max_price,mean_price,range_pct,direction_changes,avg_volatility,max_volatility,avg_momentum,avg_volume,avg_volume_spike,avg_change_pct,dir_change_norm,swing_score
symbol,,,,,,,,,,,,,
NOW,88.4300,807.450,167.148047,4.301695,37,0.817081,0.992126,0.548381,0.778683,0.702254,1.962911,0.956924,2.072889
AAOI,29.3500,157.320,40.073093,3.193415,3,0.769909,1.000000,0.938038,0.837764,1.214899,11.247184,0.847252,1.651517
SNDK,202.4600,952.500,281.561986,2.663854,137,0.963948,1.000000,0.835465,0.750665,0.830033,7.267754,0.993129,1.535677
LITE,308.2800,906.675,351.885537,1.700539,3,0.982409,1.000000,0.859667,0.606852,0.701771,6.113057,0.847252,1.136874
CORT,33.6418,84.415,37.616559,1.349757,37,0.846701,1.000000,0.411501,0.962371,9.770362,-22.230554,0.956924,1.073188
RGC,11.5300,53.165,21.472430,1.938998,1,0.639472,0.988732,0.928786,0.879804,2.407310,13.431482,0.412526,1.062272
CAR,91.8900,448.980,211.511019,1.688281,1,0.933697,1.000000,0.900130,0.926402,2.292026,9.675148,0.412526,1.040356
BE,80.7100,219.030,108.142664,1.279051,25,0.903915,0.997647,0.905039,0.671365,0.741991,8.296968,0.933404,1.009177
HYMC,24.7164,57.690,31.782749,1.037468,25,0.835769,0.957895,0.706774,0.949839,2.882637,9.961438,0.933404,0.952765


In [33]:
import plotly.express as px
import os

# -----------------------------
# TOP 20 ONLY (most important change)
# -----------------------------
df_plot = swing_candidates.head(20).copy()

# ensure clean ordering (best first visually)
df_plot = df_plot.sort_values("swing_score", ascending=False)

fig = px.scatter(
    df_plot,
    x="avg_volume_spike",
    y="avg_change_pct",
    size="range_pct",
    color="avg_volatility",
    text=df_plot.index,   # show tickers directly
    hover_name=df_plot.index,
    color_continuous_scale="Blues",
    size_max=30
)

# -----------------------------
# Make it modern + readable
# -----------------------------
fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(width=1, color="white"))
)

fig.update_layout(
    title="Top 20 Swing Stocks (High-Confidence Set)",
    xaxis_title="Volume Spike",
    yaxis_title="Price Change %",
    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",

    height=650,
    margin=dict(l=40, r=40, t=60, b=40)
)
 
chart_path = os.path.join(OUTPUT_DIR, "top_20_swing_radar.html")
fig.write_html(chart_path, include_plotlyjs="cdn")
 

In [31]:
import plotly.express as px
import os

df_bar = swing_candidates.head(100).copy()

# ensure symbol exists (sometimes it's index)
df_bar = df_bar.reset_index()

if "index" in df_bar.columns:
    df_bar = df_bar.rename(columns={"index": "symbol"})

# sort for proper ranking display
df_bar = df_bar.sort_values("swing_score", ascending=True)

fig = px.bar(
    df_bar,
    y="symbol",
    x="swing_score",
    orientation="h",
    color="avg_volatility",
    color_continuous_scale="Greens",
    hover_data=[
        "range_pct",
        "direction_changes",
        "avg_volume",
        "avg_volume_spike"
    ],
    text="swing_score"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")

fig.update_layout(
    title="Top 100 Swing Candidates (Ranked by Swing Score)",
    xaxis_title="Swing Score",
    yaxis_title="Symbol",
    plot_bgcolor="black",
    paper_bgcolor="black",
    font_color="white",
    height=900
) 
 
chart_paths = os.path.join(OUTPUT_DIR, "swing_candidates_bar.html")
fig.write_html(chart_paths, include_plotlyjs="cdn")

In [32]:
import plotly.express as px
import plotly.graph_objects as go
import os

df_bar = swing_candidates.head(100).copy().reset_index()

if "index" in df_bar.columns:
    df_bar = df_bar.rename(columns={"index": "symbol"})

df_bar = df_bar.sort_values("swing_score", ascending=True)

# 90th percentile threshold
threshold = df_bar["swing_score"].quantile(0.9)

fig = px.bar(
    df_bar,
    y="symbol",
    x="swing_score",
    orientation="h",
    color="avg_volatility",
    color_continuous_scale="Blues",  # 🔵 DARK BLUE SCALE
    hover_data=["range_pct", "direction_changes", "avg_volume"]
)

# Threshold line
fig.add_vline(
    x=threshold,
    line_width=2,
    line_dash="dash",
    line_color="white"
)

fig.add_annotation(
    x=threshold,
    y=1.02,
    xref="x",
    yref="paper",
    text="90th percentile",
    showarrow=False,
    font=dict(color="white")
)

# Dark theme styling
fig.update_layout(
    title="Top Swing Candidates (Dark Blue Theme)",
    xaxis_title="Swing Score",
    yaxis_title="Symbol",
    plot_bgcolor="#0b0f1a",   # deep navy background
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=900
)
 

chart_pathx = os.path.join(OUTPUT_DIR, "swing_bar_dark_blue.html")
fig.write_html(chart_pathx, include_plotlyjs="cdn")